# Audit DB Advanced: Views, Triggers, Hash Chains & RBAC

## Enforcing Access Control and Detecting Tampering

**Difficulty: Advanced | ~50 min | Requires Lab 3 (Querying Across the Hierarchy with JOINs) done in this Supabase project**

*Lab 4 of 5 in the Audit DB Labs module.*

Lab 3 proved that JOINs can reconstruct the full run -> span -> tool_call / guardrail_event hierarchy in a single round trip, but the query rewrites the same flattening logic every time. It also proved nothing about **integrity**: a user with database access could silently UPDATE an event's payload or DELETE a row and no one would know.

This lab solves both problems. First, a **SQL VIEW** (`v_audit_trail`) encodes the JOIN once so every downstream query reads from one canonical flattened source. Second, an **append-only trigger** on the `event` table blocks any UPDATE or DELETE. Third, a **hash chain** computes an `md5` fingerprint over each event's content and links it to the previous hash, so even a privileged user who bypasses the trigger cannot alter data without breaking the chain. Finally, a **read-only auditor role** demonstrates how `GRANT` / `REVOKE` restricts who can read the audit trail.

Everything runs against your real Supabase Postgres, so every id and timestamp you see is genuine.

In [ ]:
!pip install python-dotenv==1.2.3 psycopg2-binary==2.9.12

(pip install skipped in capture)

One pinned line installs everything the lab needs, matching Section 9 exactly.

In [ ]:
import os
import re
from hashlib import md5 as md5_fn
from pathlib import Path

from dotenv import load_dotenv
import psycopg2

# Walk upward to find the module-level .env (same pattern as Labs 1-3)
env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")
assert database_url, "DATABASE_URL not found - complete README Section 7 setup first."

# Connect with a short timeout so hanging credentials fail fast
connection = psycopg2.connect(database_url, connect_timeout=10)
cursor = connection.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])

PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit

Same connection pattern as Labs 1-3: walk upward for the module-level `.env`, load `DATABASE_URL`, connect with `psycopg2`, and print the version string as proof the connection works. We add `hashlib` and `re` here since later steps need them.

In [ ]:
# Fail-fast: confirm all prerequisite tables exist before building on them
for table in ["run", "event", "span", "tool_call", "guardrail_event"]:
    cursor.execute("SELECT to_regclass(%s)", (f"public.{table}",))
    assert cursor.fetchone()[0] is not None, f"{table} table missing - complete prior labs first."
print("Lab 1-3 schema found: run, event, span, tool_call, guardrail_event.")

Lab 1-3 schema found: run, event, span, tool_call, guardrail_event.

Same fail-fast pattern Labs 2 and 3 used. This lab needs all five tables - `event` in particular, because the append-only trigger and hash chain protect it.

In [ ]:
# --- Create the full run hierarchy (same shape as Labs 2-3) ---
cursor.execute("INSERT INTO run (agent_name, status) VALUES (%s, %s) RETURNING run_id",
               ("support-agent", "running"))
lab4_run_id = cursor.fetchone()[0]
cursor.execute("INSERT INTO span (run_id, span_name) VALUES (%s, %s) RETURNING span_id",
               (lab4_run_id, "retrieve_orders"))
span1 = cursor.fetchone()[0]
cursor.execute(
    "INSERT INTO tool_call (span_id, tool_name, arguments, result, success) VALUES (%s,%s,%s,%s,%s)",
    (span1, "query_orders_db", '{"table":"orders"}', "1284 rows returned", True))
cursor.execute("UPDATE span SET ended_at = now() WHERE span_id = %s", (span1,))
cursor.execute("INSERT INTO span (run_id, span_name) VALUES (%s, %s) RETURNING span_id",
               (lab4_run_id, "generate_answer"))
span2 = cursor.fetchone()[0]
cursor.execute(
    "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) VALUES (%s,%s,%s,%s)",
    (span2, "pii_scan", "pass", "no personal data found"))
cursor.execute(
    "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) VALUES (%s,%s,%s,%s)",
    (span2, "toxicity_scan", "fail", "flagged content detected"))
cursor.execute("UPDATE span SET ended_at = now() WHERE span_id = %s", (span2,))
cursor.execute("UPDATE run SET ended_at = now(), status = 'completed', total_cost = %s WHERE run_id = %s",
               (0.0073, lab4_run_id))
# Insert 3 events — this table is the target of append-only + hash chain
cursor.execute("INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s) RETURNING event_id",
               (lab4_run_id, "user_message", "User asked: how many orders shipped yesterday?"))
ev1 = cursor.fetchone()[0]
cursor.execute("INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s) RETURNING event_id",
               (lab4_run_id, "db_query", "Counted 1284 shipped orders."))
ev2 = cursor.fetchone()[0]
cursor.execute("INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s) RETURNING event_id",
               (lab4_run_id, "model_answer", "Answered: 1284 orders shipped yesterday."))
ev3 = cursor.fetchone()[0]
connection.commit()
print(f"Run {lab4_run_id} inserted with 2 spans, 1 tool_call, 2 guardrail_events, 3 events.")

Run 167 inserted with 2 spans, 1 tool_call, 2 guardrail_events, 3 events.

Same data shape as Labs 2-3 plus three `event` rows - the `event` table is the target of the append-only trigger and hash chain, so we need fresh rows to protect.

In [ ]:
# Create the audit view — same four-table LEFT JOIN from Lab 3, stored as a named query
cursor.execute("DROP VIEW IF EXISTS v_audit_trail")
cursor.execute("""
    CREATE VIEW v_audit_trail AS
    SELECT r.run_id, r.agent_name, r.status, r.total_cost,
           r.started_at AS run_started, r.ended_at AS run_ended,
           s.span_id, s.span_name, s.started_at AS span_started, s.ended_at AS span_ended,
           tc.tool_call_id, tc.tool_name, tc.result AS tool_result, tc.success,
           ge.guardrail_event_id, ge.check_name, ge.outcome, ge.reason
    FROM run r
    LEFT JOIN span s ON s.run_id = r.run_id
    LEFT JOIN tool_call tc ON tc.span_id = s.span_id
    LEFT JOIN guardrail_event ge ON ge.span_id = s.span_id
    ORDER BY r.run_id, s.started_at, tc.tool_call_id, ge.guardrail_event_id
""")
connection.commit()
print("View v_audit_trail created.")

cursor.execute("SELECT run_id, agent_name, span_name, tool_name, check_name, outcome "
               "FROM v_audit_trail WHERE run_id = %s", (lab4_run_id,))
for r in cursor.fetchall():
    print(f"  run {r[0]} | {r[1]} | span: {r[2]} | tool: {r[3]} | check: {r[4]} -> {r[5]}")

View v_audit_trail created.  run 167 | support-agent | span: retrieve_orders | tool: query_orders_db | check: None -> None  run 167 | support-agent | span: generate_answer | tool: None | check: pii_scan -> pass  run 167 | support-agent | span: generate_answer | tool: None | check: toxicity_scan -> fail

The same four-table LEFT JOIN from Lab 3, stored as a named view. Every downstream query now reads from `v_audit_trail` instead of rewriting the JOIN. The view re-runs the JOIN on every access, always reflecting the current data.

In [ ]:
# Append-only trigger: RAISE EXCEPTION on any UPDATE or DELETE
cursor.execute("""
    CREATE OR REPLACE FUNCTION fn_prevent_event_tamper()
    RETURNS trigger AS $$
    BEGIN
        RAISE EXCEPTION 'Audit log is append-only: % on event table is not allowed', TG_OP;
        RETURN NULL;
    END; $$ LANGUAGE plpgsql
""")
cursor.execute("DROP TRIGGER IF EXISTS trg_prevent_event_tamper ON event")
cursor.execute("""CREATE TRIGGER trg_prevent_event_tamper
    BEFORE UPDATE OR DELETE ON event FOR EACH ROW
    EXECUTE FUNCTION fn_prevent_event_tamper()""")
connection.commit()
print("Trigger trg_prevent_event_tamper created - blocks UPDATE and DELETE on event.")

try:
    cursor.execute("UPDATE event SET payload = 'should fail' WHERE event_id = %s", (ev1,))
except Exception as e:
    connection.rollback()
    print(f"UPDATE rejected: {type(e).__name__}: {e}")

Trigger trg_prevent_event_tamper created - blocks UPDATE and DELETE on event.UPDATE rejected: RaiseException: Audit log is append-only: UPDATE on event table is not allowedCONTEXT:  PL/pgSQL function fn_prevent_event_tamper() line 3 at RAISE

The trigger function does one thing: `RAISE EXCEPTION` on every UPDATE or DELETE. The demo proves the trigger fires - the UPDATE is rejected before any row is modified. This is the first layer of defense: the `event` table is append-only by construction.

In [ ]:
# Create the hash chain table and auto-hash trigger
cursor.execute("DROP TABLE IF EXISTS event_hash_chain")
cursor.execute("""CREATE TABLE event_hash_chain (
    chain_id SERIAL PRIMARY KEY,
    event_id INTEGER NOT NULL REFERENCES event(event_id),
    row_hash TEXT NOT NULL,
    prev_hash TEXT,
    chained_at TIMESTAMPTZ DEFAULT now()
)""")
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_chain_event_unique ON event_hash_chain (event_id)")
cursor.execute("""CREATE OR REPLACE FUNCTION fn_auto_hash_event() RETURNS trigger AS $$
    DECLARE last_hash TEXT; content TEXT;
    BEGIN
        SELECT row_hash INTO last_hash FROM event_hash_chain ORDER BY chain_id DESC LIMIT 1;
        content := COALESCE(NEW.event_id::text,'') || '|' || COALESCE(NEW.run_id::text,'') || '|' ||
                   COALESCE(NEW.event_type,'') || '|' || COALESCE(NEW.payload,'') || '|' ||
                   COALESCE(last_hash,'GENESIS');
        INSERT INTO event_hash_chain (event_id, row_hash, prev_hash)
        VALUES (NEW.event_id, md5(content), last_hash);
        RETURN NEW;
    END; $$ LANGUAGE plpgsql""")
cursor.execute("DROP TRIGGER IF EXISTS trg_auto_hash_event ON event")
cursor.execute("""CREATE TRIGGER trg_auto_hash_event
    AFTER INSERT ON event FOR EACH ROW
    EXECUTE FUNCTION fn_auto_hash_event()""")
connection.commit()
print("Hash chain table and auto-hash trigger created.")

Hash chain table and auto-hash trigger created.

The `event_hash_chain` table stores one row per event: `row_hash` (the md5 fingerprint) and `prev_hash` (the hash of the event that came before it). The Python-driven loop in the next cell populates it for existing events. `GENESIS` is the sentinel for the first event in a run.

In [ ]:
# Backfill: existing events from setup need manual chain links
cursor.execute("DELETE FROM event_hash_chain WHERE event_id IN "
               "(SELECT event_id FROM event WHERE run_id = %s)", (lab4_run_id,))
cursor.execute("SELECT event_id, run_id, event_type, payload FROM event "
               "WHERE run_id = %s ORDER BY event_id", (lab4_run_id,))
# Compute hash = md5(event_id|run_id|event_type|payload|prev_hash)
prev_hash = None
for eid, rid, etype, payload in cursor.fetchall():
    content = f"{eid}|{rid}|{etype}|{payload}|{prev_hash or 'GENESIS'}"
    row_hash = md5_fn(content.encode()).hexdigest()
    cursor.execute("INSERT INTO event_hash_chain (event_id, row_hash, prev_hash) VALUES (%s, %s, %s)",
                   (eid, row_hash, prev_hash))
    prev_hash = row_hash
connection.commit()

# Verify: recompute every hash and compare to stored value
def verify_hash_chain(conn, run_id):
    cur = conn.cursor()
    cur.execute("""
        SELECT ec.event_id, ec.row_hash, ec.prev_hash, e.run_id, e.event_type, e.payload
        FROM event_hash_chain ec JOIN event e ON e.event_id = ec.event_id
        WHERE e.run_id = %s ORDER BY ec.chain_id""", (run_id,))
    data = cur.fetchall(); cur.close()
    cb = lb = 0; prev_s = None
    for eid, stored, prev, rid, etype, payload in data:
        c = f"{eid}|{rid}|{etype}|{payload}|{prev or 'GENESIS'}"
        if stored != md5_fn(c.encode()).hexdigest(): cb += 1  # content break
        if prev_s is not None and prev != prev_s: lb += 1     # linkage break
        prev_s = stored
    return cb, lb, len(data)

cb, lb, total = verify_hash_chain(connection, lab4_run_id)
print(f"Chain verification: {cb} content break(s), {lb} linkage break(s) across {total} links.")

Chain verification: 0 content break(s), 0 linkage break(s) across 3 links.

Verification recomputes every hash from the live `event` data and compares to the stored value. **Content breaks** mean the event data was altered after chaining. **Chain linkage breaks** mean a `prev_hash` doesn't match the stored hash of the preceding event. Zero on both counts means the chain is intact.

In [ ]:
# Simulate privileged admin bypass: drop trigger -> tamper -> re-create trigger
cursor.execute("DROP TRIGGER trg_prevent_event_tamper ON event"); connection.commit()
cursor.execute("SELECT payload FROM event WHERE event_id = %s", (ev2,))
old_payload = cursor.fetchone()[0]
cursor.execute("UPDATE event SET payload = %s WHERE event_id = %s",
               ("TAMPERED: wrong count, was actually 9999.", ev2))
connection.commit()
cursor.execute("SELECT payload FROM event WHERE event_id = %s", (ev2,))
new_payload = cursor.fetchone()[0]
cursor.execute("""CREATE TRIGGER trg_prevent_event_tamper
    BEFORE UPDATE OR DELETE ON event FOR EACH ROW
    EXECUTE FUNCTION fn_prevent_event_tamper()""")
connection.commit()
print(f"Tampered event {ev2}:\n  Before: {old_payload}\n  After:  {new_payload}")

# Verify — hash chain should detect the content break
cb, lb, total = verify_hash_chain(connection, lab4_run_id)
print(f"Chain verification: {cb} content break(s), {lb} linkage break(s) across {total} links.")

# Restore original payload and rebuild the chain
cursor.execute("DROP TRIGGER trg_prevent_event_tamper ON event"); connection.commit()
cursor.execute("UPDATE event SET payload = %s WHERE event_id = %s", (old_payload, ev2))
connection.commit()
cursor.execute("""CREATE TRIGGER trg_prevent_event_tamper
    BEFORE UPDATE OR DELETE ON event FOR EACH ROW
    EXECUTE FUNCTION fn_prevent_event_tamper()""")
connection.commit()
# Rebuild chain links for this run
cursor.execute("DELETE FROM event_hash_chain WHERE event_id IN "
               "(SELECT event_id FROM event WHERE run_id = %s)", (lab4_run_id,))
cursor.execute("SELECT event_id, run_id, event_type, payload FROM event "
               "WHERE run_id = %s ORDER BY event_id", (lab4_run_id,))
prev_hash = None
for eid, rid, etype, payload in cursor.fetchall():
    content = f"{eid}|{rid}|{etype}|{payload}|{prev_hash or 'GENESIS'}"
    row_hash = md5_fn(content.encode()).hexdigest()
    cursor.execute("INSERT INTO event_hash_chain (event_id, row_hash, prev_hash) VALUES (%s, %s, %s)",
                   (eid, row_hash, prev_hash))
    prev_hash = row_hash
connection.commit()
print(f'Payload restored, chain rebuilt. Original value: "{old_payload}"')

Tampered event 133:  Before: Counted 1284 shipped orders.  After:  TAMPERED: wrong count, was actually 9999.Chain verification: 1 content break(s), 0 linkage break(s) across 3 links.Payload restored, chain rebuilt. Original value: "Counted 1284 shipped orders."

This is the critical demonstration. The append-only trigger blocks normal clients, but a privileged admin can drop the trigger, alter data, and re-create the trigger. The hash chain is the only layer that catches this: after tampering, verification finds a content break. After restoring the original payload and rebuilding the chain, verification returns to 0 breaks.

In [ ]:
# Clean up any previous auditor user (ignore errors if it doesn't exist)
for stmt in [
    "REVOKE ALL ON SCHEMA public FROM lab4_auditor",
    "REVOKE ALL ON ALL TABLES IN SCHEMA public FROM lab4_auditor",
    "DROP USER IF EXISTS lab4_auditor",
]:
    try: cursor.execute(stmt)
    except Exception: connection.rollback()
connection.commit()
# Create the auditor role with SELECT-only access on the view
cursor.execute("CREATE USER lab4_auditor WITH PASSWORD 'lab4_test_pass_2026'")
cursor.execute("GRANT SELECT ON v_audit_trail TO lab4_auditor")
cursor.execute("GRANT USAGE ON SCHEMA public TO lab4_auditor")
connection.commit()
print("User lab4_auditor created with SELECT on v_audit_trail only.")
# Verify permissions via information_schema
cursor.execute("""SELECT grantee, table_name, privilege_type FROM information_schema.role_table_grants
    WHERE grantee = 'lab4_auditor' ORDER BY table_name, privilege_type""")
for g, t, p in cursor.fetchall(): print(f"  {g} | {t} | {p}")

# Cleanup: drop all Lab 4 objects and sample data
for stmt in [
    "DROP TRIGGER IF EXISTS trg_prevent_event_tamper ON event",
    "DROP FUNCTION IF EXISTS fn_prevent_event_tamper()",
    "DROP TRIGGER IF EXISTS trg_auto_hash_event ON event",
    "DROP FUNCTION IF EXISTS fn_auto_hash_event()",
    "DROP TABLE IF EXISTS event_hash_chain",
    "DROP VIEW IF EXISTS v_audit_trail",
]:
    try: cursor.execute(stmt)
    except Exception: connection.rollback()
# Drop the auditor role so it doesn't outlive the lab
for stmt in [
    "REVOKE ALL ON SCHEMA public FROM lab4_auditor",
    "REVOKE ALL ON ALL TABLES IN SCHEMA public FROM lab4_auditor",
    "DROP USER IF EXISTS lab4_auditor",
]:
    try: cursor.execute(stmt)
    except Exception: connection.rollback()
# Delete sample data in child-first order
cursor.execute("DELETE FROM guardrail_event WHERE span_id IN "
               "(SELECT span_id FROM span WHERE run_id = %s)", (lab4_run_id,))
cursor.execute("DELETE FROM tool_call WHERE span_id IN "
               "(SELECT span_id FROM span WHERE run_id = %s)", (lab4_run_id,))
cursor.execute("DELETE FROM span WHERE run_id = %s", (lab4_run_id,))
cursor.execute("DELETE FROM event WHERE run_id = %s", (lab4_run_id,))
cursor.execute("DELETE FROM run WHERE run_id = %s", (lab4_run_id,))
connection.commit()
cursor.close(); connection.close()
print(f"Run {lab4_run_id}: views + trigger + hash chain + auditor role - lab complete.")

User lab4_auditor created with SELECT on v_audit_trail only.  lab4_auditor | v_audit_trail | SELECTRun 167: views + trigger + hash chain + auditor role - lab complete.